# LiDAR + SAM3D Body OBB Visualization

Loads the LiDAR point cloud and the oriented bounding boxes (OBBs) produced by
`run_sam3d_bodies.py` for a given scene/frame and visualizes them together in 3D
using **plotly**.

Coordinate frame: **LiDAR sensor frame**.

Transform chain:
```
body-relative + cam_t  →  camera frame  →  ego frame  →  LiDAR frame
```

In [8]:
from pathlib import Path

DATA_ROOT          = "/media/lleba/ECP_Nuscenes_01/output2/ecp2nuscenes"
NUSCENES_SCENE_IDX = 11
FRAME_IDX          = 350   # must have been processed by run_sam3d_bodies.py

SAM3D_OUTPUT_DIR = Path(
    f"/media/lleba/ECP_Nuscenes_01/SAM3D_Outputs/Strassbourg/"
    f"scene_{NUSCENES_SCENE_IDX:03d}/SAM3D_Body"
)

LIDAR_STRIDE = 1   # keep every Nth point (reduce for speed; 1 = all ~75 k pts)

In [9]:
import numpy as np
from pyquaternion import Quaternion
from nuscenes.nuscenes import NuScenes

print("Loading nuScenes...")
nusc  = NuScenes(version="v1.0-trainval", dataroot=DATA_ROOT, verbose=False)
scene = nusc.scene[NUSCENES_SCENE_IDX]
print(f"  Scene: {scene['name']}")

In [10]:
def get_frame_info(nusc, scene, frame_idx):
    token = nusc.get("sample", scene["first_sample_token"])["data"]["CAM_FRONT"]
    for i in range(frame_idx + 1):
        rec = nusc.get("sample_data", token)
        if i == frame_idx:
            sample      = nusc.get("sample", rec["sample_token"])
            cam_calib   = nusc.get("calibrated_sensor", rec["calibrated_sensor_token"])
            lidar_sd    = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
            lidar_calib = nusc.get("calibrated_sensor", lidar_sd["calibrated_sensor_token"])
            lidar_path  = nusc.get_sample_data_path(sample["data"]["LIDAR_TOP"])
            return rec, cam_calib, lidar_calib, lidar_path
        token = rec["next"]
        if not token:
            raise IndexError(f"Frame {frame_idx} out of range")

cam_sd, cam_calib, lidar_calib, lidar_path = get_frame_info(nusc, scene, FRAME_IDX)

R_cam   = Quaternion(cam_calib["rotation"]).rotation_matrix
t_cam   = np.array(cam_calib["translation"],   dtype=np.float64)
R_lidar = Quaternion(lidar_calib["rotation"]).rotation_matrix
t_lidar = np.array(lidar_calib["translation"], dtype=np.float64)

print(f"CAM_FRONT  t={t_cam.round(3)}")
print(f"LIDAR_TOP  t={t_lidar.round(3)}")

In [11]:
def cam_to_lidar(pts_cam):
    """(N,3) camera frame → (N,3) LiDAR frame."""
    pts_ego   = (R_cam @ pts_cam.T).T + t_cam
    return (R_lidar.T @ (pts_ego - t_lidar).T).T

def load_bbox_in_lidar(ped_dir):
    bbox_3d = np.load(ped_dir / "bbox_3d.npy")          # (8,3) body-relative
    cam_t   = np.load(ped_dir / "cam_t.npy")            # (3,)
    return cam_to_lidar((bbox_3d + cam_t).astype(np.float64))

In [12]:
pts_raw   = np.fromfile(lidar_path, dtype=np.float32).reshape(-1, 5)
xyz       = pts_raw[::LIDAR_STRIDE, :3].astype(np.float64)
intensity = pts_raw[::LIDAR_STRIDE, 3].astype(np.float32)
print(f"LiDAR: {len(xyz):,} points (stride={LIDAR_STRIDE})")

In [13]:
BBOX_EDGES = [
    (0,1),(1,2),(2,3),(3,0),  # back face
    (4,5),(5,6),(6,7),(7,4),  # front face
    (0,4),(1,5),(2,6),(3,7),  # pillars
]

ped_dirs = sorted(SAM3D_OUTPUT_DIR.glob(f"frame{FRAME_IDX:05d}_ped*"))
print(f"Found {len(ped_dirs)} pedestrian(s) for frame {FRAME_IDX}")

all_corners = []
for d in ped_dirs:
    c = load_bbox_in_lidar(d)
    all_corners.append(c)
    h = c[:,2].max() - c[:,2].min()
    print(f"  {d.name}: center={c.mean(0).round(2)}  H={h:.2f}m")

In [14]:
import plotly.graph_objects as go

PALETTE = [
    "rgb(255,69,0)",   # orange-red
    "rgb(0,255,127)",  # spring green
    "rgb(30,144,255)", # dodger blue
    "rgb(255,215,0)",  # gold
    "rgb(255,105,180)",# hot pink
    "rgb(0,206,209)",  # dark turquoise
]

traces = []

# ── LiDAR point cloud (height-coloured) ───────────────────────────────────────
traces.append(go.Scatter3d(
    x=xyz[:,0], y=xyz[:,1], z=xyz[:,2],
    mode="markers",
    marker=dict(
        size=1,
        color=xyz[:,2],
        colorscale="Turbo",
        opacity=0.5,
        showscale=False,
    ),
    name="LiDAR",
    hoverinfo="skip",
))

# ── OBB wireframes ────────────────────────────────────────────────────────────
for ped_idx, corners in enumerate(all_corners):
    colour = PALETTE[ped_idx % len(PALETTE)]

    # Collect edge segments (separated by None for plotly line breaks)
    xs, ys, zs = [], [], []
    for a, b in BBOX_EDGES:
        xs += [corners[a,0], corners[b,0], None]
        ys += [corners[a,1], corners[b,1], None]
        zs += [corners[a,2], corners[b,2], None]

    traces.append(go.Scatter3d(
        x=xs, y=ys, z=zs,
        mode="lines",
        line=dict(color=colour, width=4),
        name=f"OBB ped {ped_idx:02d}",
    ))

    # ── Facing arrow ──────────────────────────────────────────────────────────
    d      = ped_dirs[ped_idx]
    joints = np.load(d / "joints_3d.npy")   # (70,3) body-relative
    cam_t  = np.load(d / "cam_t.npy")       # (3,)

    _L, _R, _NOSE = 5, 6, 0
    smid = (joints[_L] + joints[_R]) / 2.0
    sv   = joints[_R] - joints[_L]
    sv_xz = np.array([sv[0], 0.0, sv[2]], dtype=np.float64)
    sv_xz /= np.linalg.norm(sv_xz) + 1e-8
    fwd = np.array([-sv_xz[2], 0.0, sv_xz[0]])
    if np.dot(fwd, joints[_NOSE] - smid) < 0:
        fwd = -fwd

    root_lid = cam_to_lidar((smid + cam_t)[np.newaxis])[0]
    tip_lid  = cam_to_lidar((smid + fwd * 0.7 + cam_t)[np.newaxis])[0]

    traces.append(go.Scatter3d(
        x=[root_lid[0], tip_lid[0]],
        y=[root_lid[1], tip_lid[1]],
        z=[root_lid[2], tip_lid[2]],
        mode="lines+markers",
        line=dict(color=colour, width=6),
        marker=dict(size=[2, 6], color=colour, symbol=["circle", "diamond"]),
        name=f"Facing ped {ped_idx:02d}",
        showlegend=False,
    ))

fig = go.Figure(data=traces)
fig.update_layout(
    title=f"Scene {NUSCENES_SCENE_IDX} / Frame {FRAME_IDX} — LiDAR + OBBs",
    scene=dict(
        xaxis_title="X (m)",
        yaxis_title="Y (m)",
        zaxis_title="Z (m)",
        aspectmode="data",         # keeps metric proportions
        camera=dict(
            eye=dict(x=0, y=-0.5, z=1.5),  # bird's-eye-ish view
        ),
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    height=700,
)
fig.show()

## Multi-frame view

OBBs from several processed frames overlaid on the same LiDAR scan.
Each frame gets a distinct colour. Useful for a quick temporal sanity check
(note: no ego-motion compensation — LiDAR is fixed from `FRAME_IDX`).

In [ ]:
EXTRA_FRAMES  = [950, 1000, 1050]
FRAME_COLOURS = ["rgb(255,69,0)", "rgb(0,255,127)", "rgb(30,144,255)", "rgb(255,215,0)"]

traces2 = [traces[0]]  # reuse LiDAR cloud from above

for fi, frame in enumerate([FRAME_IDX] + EXTRA_FRAMES):
    colour = FRAME_COLOURS[fi % len(FRAME_COLOURS)]
    try:
        _, fc, _, _ = get_frame_info(nusc, scene, frame)
    except Exception:
        continue
    R_fc = Quaternion(fc["rotation"]).rotation_matrix
    t_fc = np.array(fc["translation"], dtype=np.float64)

    def c2l(pts, R=R_fc, t=t_fc):
        return (R_lidar.T @ ((R @ pts.T).T + t - t_lidar).T).T

    for d in sorted(SAM3D_OUTPUT_DIR.glob(f"frame{frame:05d}_ped*")):
        bbox_3d = np.load(d / "bbox_3d.npy").astype(np.float64)
        c_t     = np.load(d / "cam_t.npy").astype(np.float64)
        corners = c2l(bbox_3d + c_t)

        xs, ys, zs = [], [], []
        for a, b in BBOX_EDGES:
            xs += [corners[a,0], corners[b,0], None]
            ys += [corners[a,1], corners[b,1], None]
            zs += [corners[a,2], corners[b,2], None]

        traces2.append(go.Scatter3d(
            x=xs, y=ys, z=zs,
            mode="lines",
            line=dict(color=colour, width=4),
            name=f"f{frame} {d.name[-5:]}",
        ))

fig2 = go.Figure(data=traces2)
fig2.update_layout(
    title="Multi-frame OBBs",
    scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="data"),
    height=700,
    margin=dict(l=0, r=0, b=0, t=40),
)
fig2.show()